# Next-Word Prediction with GRUs — NumPy vs TensorFlow vs PyTorch

This notebook trains a stacked (multi-layer) **GRU** three ways and compares
them on the **same data, architecture, and hyper-parameters**:

1. **Manual (NumPy)** — implemented from scratch in
   [`gru_scratch.py`](gru_scratch.py).
2. **TensorFlow / Keras** — [`gru_tensorflow.py`](gru_tensorflow.py).
3. **PyTorch** — [`gru_pytorch.py`](gru_pytorch.py).

All three expose the same interface (`train()`, `predict()`), so the shared
helpers in [`utils.py`](utils.py) — `evaluate`, `generate` — and
[`compare.py`](compare.py)'s `compare_models` work on every model unchanged.

The task is **next-word prediction**: each input token is a dense **word
embedding**, the target is the next word (one-hot over the vocabulary), so the
model runs in `task="classification"`.

**Pipeline:** raw text → vocabulary → training sequences → train/test split →
train the three models → compare train/test accuracy → generate text.

| Section | Contents |
|---------|----------|
| 1 | Imports |
| 2 | Data |
| 3 | Next-word prediction — Word2Vec · GloVe · FastText · BERT (each a 3-way comparison) |

> **Note on the comparison.** The frameworks train with **Adam** while the
> from-scratch model uses plain **SGD + gradient clipping**, so their learned
> weights — and accuracies — differ. This is a realistic "library vs scratch"
> comparison, not a bit-for-bit match.

## 1. Imports

In [1]:
import importlib

import numpy as np

import  utils, compare, gru_pytorch, gru_tensorflow, gru_scratch

# Reload local modules so edits to the .py files are picked up without
# having to restart the kernel.
for _m in ( utils, compare, gru_pytorch, gru_tensorflow, gru_scratch):
    importlib.reload(_m)
from gru_scratch import GRU
from gru_pytorch import PyTorchGRU
from gru_tensorflow import KerasGRU
from utils import generate_dataset, train_test_split, evaluate, predict_next, generate
from compare import compare_models

In [2]:
sentences = [
'Machine Learning (ML) is a branch of Artificial Intelligence (AI)',
'that enables computers to learn from data and improve their performance',
'without being explicitly programmed. ML algorithms identify patterns in',
'historical data and use those patterns to make predictions or decisions on new data.',
'Machine learning is widely used in various applications such as recommendation systems,',
'image recognition, speech processing, fraud detection, healthcare diagnostics,',
'and autonomous vehicles. The main types of machine learning are supervised learning,',
'unsupervised learning, and reinforcement learning. As the amount of available data continues to grow,',
'machine learning plays an increasingly important role in helping organizations automate tasks,',
'gain insights, and make data-driven decisions.'
]
sentence_words = [s.split() for s in sentences]
words = [w for s in sentence_words for w in s]

## 2. Data

A small block of text to learn from. With a corpus this size the model can only
learn surface statistics, but it is enough to demonstrate and compare the GRUs.

In [3]:
def train_models(X_train, Y_train, hidden_layers, lr, epochs, batch_size, task):
    # lr is a 3-tuple (lr_manual, lr_pytorch, lr_keras). The from-scratch model
    # uses plain SGD, so its raw gradients are small and it needs a much larger
    # learning rate than the Adam-based frameworks (which normalise by gradient
    # RMS and stay effective at ~0.01). Passing one lr per model keeps the
    # comparison fair instead of leaving the manual GRU essentially frozen.
    lr_manual, lr_torch, lr_keras = lr

    print('-'*100)
    print('Training Manual GRU')
    manual_gru = GRU(X_train, Y_train, hidden_layers=hidden_layers,
                     learning_rate=lr_manual, epochs=epochs, batch_size=batch_size, task=task)
    manual_gru.train()

    print()
    print('-'*100)
    print('Training PyTorch GRU')
    torch_gru = PyTorchGRU(X_train, Y_train, hidden_layers=hidden_layers,
                         learning_rate=lr_torch, epochs=epochs, batch_size=batch_size, task=task).train()
    
    print()
    print('-'*100)
    print('Training TensorFlow GRU')
    keras_gru = KerasGRU(X_train, Y_train, hidden_layers=hidden_layers,
                         learning_rate=lr_keras, epochs=epochs, batch_size=batch_size, task=task).train()

    print()
    print('-'*100)

    trained_models = {"manual": manual_gru,
                      "pytorch": torch_gru,
                      'tensorflow' : keras_gru
                      }

    return trained_models

## 3. Next-Word Prediction

Each input token is a dense **word embedding**; targets are one-hot over the
vocabulary, so the model runs in `task="classification"` and predicts the next
word.

Each subsection feeds the same corpus through a different embedding
(Word2Vec · GloVe · FastText · BERT) and runs the **full 3-way comparison**
(manual / TensorFlow / PyTorch).

### 3.1 Word2Vec embeddings

Train a Word2Vec model on our sentences (skip-gram) and use its vectors as the GRU's input features. This section also demonstrates a **2-layer stack** (`hidden_layers=(100, 64)`).

In [4]:
from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentence_words,
    vector_size=100,  # dimensionality of the word vectors
    window=5,        # context window size
    min_count=1,     # minimum word frequency to include in the model
    workers=4,       # number of CPU cores to use for training
    sg=1             # use skip-gram architecture (sg=0 for CBOW)
)
word_vectors = w2v_model.wv

In [5]:
input_seqs, target_seqs, vocab_to_index, index_to_vocab = generate_dataset(
    words, T_x=5, word_vectors=word_vectors)
print(f"X: {input_seqs.shape}   Y: {target_seqs.shape}   (m, T_x, vector_size)")

X_train, X_test, Y_train, Y_test = train_test_split(input_seqs, target_seqs, test_size=0.2)

Generated 102 training sequences of length 5 from a corpus of 108 words, with a vocabulary of 88 unique words.
X: (102, 5, 100)   Y: (102, 5, 88)   (m, T_x, vector_size)
Split 102 sequences -> 82 train / 20 test.


In [6]:
hidden_layers, lr, epochs, batch_size, task = (100,), (8.0, 0.03, 0.03), 15, 32, 'classification'
print('Word2Vec Models')
w2v_models = train_models(
    X_train, Y_train, hidden_layers, lr, epochs, batch_size, task
)

Word2Vec Models
----------------------------------------------------------------------------------------------------
Training Manual GRU
epoch 1/15 - loss: 4.4644
epoch 2/15 - loss: 4.4268
epoch 3/15 - loss: 4.4074
epoch 4/15 - loss: 4.4054
epoch 5/15 - loss: 4.3717
epoch 6/15 - loss: 4.3800
epoch 7/15 - loss: 4.3690
epoch 8/15 - loss: 4.3696
epoch 9/15 - loss: 4.3733
epoch 10/15 - loss: 4.3803
epoch 11/15 - loss: 4.3894
epoch 12/15 - loss: 4.3621
epoch 13/15 - loss: 4.3771
epoch 14/15 - loss: 4.3608
epoch 15/15 - loss: 4.3740

----------------------------------------------------------------------------------------------------
Training PyTorch GRU
[PyTorch] epoch 1/15, loss 4.4917
[PyTorch] epoch 2/15, loss 4.4312
[PyTorch] epoch 3/15, loss 4.3547
[PyTorch] epoch 4/15, loss 4.3023
[PyTorch] epoch 5/15, loss 4.2285
[PyTorch] epoch 6/15, loss 4.1500
[PyTorch] epoch 7/15, loss 3.9684
[PyTorch] epoch 8/15, loss 3.6685
[PyTorch] epoch 9/15, loss 3.3808
[PyTorch] epoch 10/15, loss 2.8606
[Py

In [7]:
compare_models(
    w2v_models, X_train, Y_train, X_test, Y_test,
    embedding=word_vectors, decoder=index_to_vocab,
    seed_word='Machine', num_gen=10, sample=True,
)

=== accuracy: train / test ===
  model               train     test
  manual             0.0463   0.0400
  pytorch            0.8634   0.4000
  tensorflow         0.8854   0.4100

=== generation from 'Machine' ===
  manual           Machine Learning vehicles. historical is The learning an branch continues of
  pytorch          Machine widely algorithms are fraud in in organizations automate automate learning
  tensorflow       Machine learning is an a branch of Artificial Intelligence such in


### 3.2 GloVe — pre-trained embeddings (Google-News word2vec)

Swap our small in-house vectors for Google's pre-trained 300-d word2vec vectors (trained on ~100B words of Google News).

In [8]:
# pre-trained vectors from the Google-News dataset (300d, 3M words)
import gensim.downloader as api
glove = api.load("word2vec-google-news-300")

In [9]:
input_seqs, target_seqs, vocab_to_index, index_to_vocab = generate_dataset(
    words, T_x=5, word_vectors=glove)
print(f"X: {input_seqs.shape}   Y: {target_seqs.shape}   (m, T_x, vector_size)")

X_train, X_test, Y_train, Y_test = train_test_split(input_seqs, target_seqs, test_size=0.2)

Generated 72 training sequences of length 5 from a corpus of 78 words, with a vocabulary of 67 unique words.
X: (72, 5, 300)   Y: (72, 5, 67)   (m, T_x, vector_size)
Split 72 sequences -> 58 train / 14 test.


In [10]:
hidden_layers, lr, epochs, batch_size, task = (100,), (8.0, 0.03, 0.03), 15, 32, 'classification'
print('GloVe Models')
glove_models = train_models(
    X_train, Y_train, hidden_layers, lr, epochs, batch_size, task
)

GloVe Models
----------------------------------------------------------------------------------------------------
Training Manual GRU
epoch 1/15 - loss: 4.1474
epoch 2/15 - loss: 3.5773
epoch 3/15 - loss: 2.8572
epoch 4/15 - loss: 1.8046
epoch 5/15 - loss: 1.1112
epoch 6/15 - loss: 0.7705
epoch 7/15 - loss: 0.5811
epoch 8/15 - loss: 0.3812
epoch 9/15 - loss: 0.3253
epoch 10/15 - loss: 0.2757
epoch 11/15 - loss: 0.1845
epoch 12/15 - loss: 0.1664
epoch 13/15 - loss: 0.1811
epoch 14/15 - loss: 0.1330
epoch 15/15 - loss: 0.1239

----------------------------------------------------------------------------------------------------
Training PyTorch GRU
[PyTorch] epoch 1/15, loss 4.0032
[PyTorch] epoch 2/15, loss 2.4733
[PyTorch] epoch 3/15, loss 0.8503
[PyTorch] epoch 4/15, loss 0.3075
[PyTorch] epoch 5/15, loss 0.1353
[PyTorch] epoch 6/15, loss 0.0860
[PyTorch] epoch 7/15, loss 0.0643
[PyTorch] epoch 8/15, loss 0.0508
[PyTorch] epoch 9/15, loss 0.0528
[PyTorch] epoch 10/15, loss 0.0474
[PyTor

In [11]:
compare_models(
    glove_models, X_train, Y_train, X_test, Y_test,
    embedding=glove, decoder=index_to_vocab,
    seed_word='Machine', num_gen=10, sample=True,
)

=== accuracy: train / test ===
  model               train     test
  manual             0.9759   0.9571
  pytorch            0.9759   0.9714
  tensorflow         0.9759   0.9714

=== generation from 'Machine' ===
  manual           Machine Learning is widely used in various applications such as recommendation
  pytorch          Machine learning is widely used in various applications such as recommendation
  tensorflow       Machine learning is widely used in various applications such as recommendation


### 3.3 FastText embeddings

FastText builds word vectors from character n-grams, so it can embed even rare or out-of-vocabulary words. Trained here on our own sentences.

In [12]:
from gensim.models import FastText

ft_model = FastText(
    sentence_words,
    vector_size=150,  # dimensionality of the word vectors
    window=5,        # context window size
    min_count=1,     # minimum word frequency to include in the model
    workers=4,       # number of CPU cores to use for training
    sg=1             # use skip-gram architecture (sg=0 for CBOW)
)
fasttext_vectors = ft_model.wv

In [13]:
input_seqs, target_seqs, vocab_to_index, index_to_vocab = generate_dataset(
    words, T_x=5, word_vectors=fasttext_vectors)
print(f"X: {input_seqs.shape}   Y: {target_seqs.shape}   (m, T_x, vector_size)")

X_train, X_test, Y_train, Y_test = train_test_split(input_seqs, target_seqs, test_size=0.2)

Generated 102 training sequences of length 5 from a corpus of 108 words, with a vocabulary of 88 unique words.
X: (102, 5, 150)   Y: (102, 5, 88)   (m, T_x, vector_size)
Split 102 sequences -> 82 train / 20 test.


In [14]:

hidden_layers, lr, epochs, batch_size, task = (50,), (8.0, 0.03, 0.03), 15, 32, 'classification'
print('FastText Models')
fasttext_models = train_models(
    X_train, Y_train, hidden_layers, lr, epochs, batch_size, task
)

FastText Models
----------------------------------------------------------------------------------------------------
Training Manual GRU
epoch 1/15 - loss: 4.4652
epoch 2/15 - loss: 4.4111
epoch 3/15 - loss: 4.4110
epoch 4/15 - loss: 4.3896
epoch 5/15 - loss: 4.3791
epoch 6/15 - loss: 4.4103
epoch 7/15 - loss: 4.3736
epoch 8/15 - loss: 4.3813
epoch 9/15 - loss: 4.3582
epoch 10/15 - loss: 4.3646
epoch 11/15 - loss: 4.3528
epoch 12/15 - loss: 4.3684
epoch 13/15 - loss: 4.3627
epoch 14/15 - loss: 4.3691
epoch 15/15 - loss: 4.3526

----------------------------------------------------------------------------------------------------
Training PyTorch GRU
[PyTorch] epoch 1/15, loss 4.4902
[PyTorch] epoch 2/15, loss 4.3988
[PyTorch] epoch 3/15, loss 4.3683
[PyTorch] epoch 4/15, loss 4.3469
[PyTorch] epoch 5/15, loss 4.3493
[PyTorch] epoch 6/15, loss 4.3230
[PyTorch] epoch 7/15, loss 4.3069
[PyTorch] epoch 8/15, loss 4.3025
[PyTorch] epoch 9/15, loss 4.2993
[PyTorch] epoch 10/15, loss 4.2461
[Py

In [15]:
compare_models(
    fasttext_models, X_train, Y_train, X_test, Y_test,
    embedding=fasttext_vectors, decoder=index_to_vocab,
    seed_word='Machine', num_gen=10, sample=True,
)

=== accuracy: train / test ===
  model               train     test
  manual             0.0463   0.0400
  pytorch            0.1024   0.0100
  tensorflow         0.1634   0.0600

=== generation from 'Machine' ===
  manual           Machine ML speech use and branch role The recognition, helping patterns
  pytorch          Machine learning (AI) branch role make predictions available types enables unsupervised
  tensorflow       Machine The Artificial in various explicitly helping amount predictions patterns vehicles.


### 3.4 BERT embeddings

Use a pre-trained BERT model to produce a contextual `[CLS]` embedding per word, then feed those 768-d vectors to the GRUs.

In [16]:
from transformers import BertTokenizer, BertModel
import torch

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')

bert_word_to_vec = {}
for word in words:
    inputs = tokenizer(word, return_tensors="pt")
    with torch.no_grad():
        outputs = bert_model(**inputs)
    bert_word_to_vec[word] = outputs.last_hidden_state[:, 0, :].squeeze().numpy()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
input_seqs, target_seqs, vocab_to_index, index_to_vocab = generate_dataset(
    words, T_x=5, word_vectors=bert_word_to_vec)
print(f"X: {input_seqs.shape}   Y: {target_seqs.shape}   (m, T_x, vector_size)")
X_train, X_test, Y_train, Y_test = train_test_split(input_seqs, target_seqs, test_size=0.2)

Generated 102 training sequences of length 5 from a corpus of 108 words, with a vocabulary of 88 unique words.
X: (102, 5, 768)   Y: (102, 5, 88)   (m, T_x, vector_size)
Split 102 sequences -> 82 train / 20 test.


In [18]:
hidden_layers, lr, epochs, batch_size, task = (100,), (8.0, 0.03, 0.03), 15, 32, 'classification'
print('BERT Models')
bert_models = train_models(
    X_train, Y_train, hidden_layers, lr, epochs, batch_size, task
)

BERT Models
----------------------------------------------------------------------------------------------------
Training Manual GRU
epoch 1/15 - loss: 5.5305
epoch 2/15 - loss: 7.8444
epoch 3/15 - loss: 8.4515
epoch 4/15 - loss: 9.8094
epoch 5/15 - loss: 10.4949
epoch 6/15 - loss: 10.8737
epoch 7/15 - loss: 10.2349
epoch 8/15 - loss: 9.1941
epoch 9/15 - loss: 9.0494
epoch 10/15 - loss: 9.8772
epoch 11/15 - loss: 9.5307
epoch 12/15 - loss: 8.0410
epoch 13/15 - loss: 9.3526
epoch 14/15 - loss: 9.0964
epoch 15/15 - loss: 9.2652

----------------------------------------------------------------------------------------------------
Training PyTorch GRU
[PyTorch] epoch 1/15, loss 4.5585
[PyTorch] epoch 2/15, loss 4.2402
[PyTorch] epoch 3/15, loss 3.7851
[PyTorch] epoch 4/15, loss 3.4005
[PyTorch] epoch 5/15, loss 2.9423
[PyTorch] epoch 6/15, loss 2.4455
[PyTorch] epoch 7/15, loss 2.0221
[PyTorch] epoch 8/15, loss 1.5791
[PyTorch] epoch 9/15, loss 1.2541
[PyTorch] epoch 10/15, loss 0.9850
[PyT

In [19]:
compare_models(
    bert_models, X_train, Y_train, X_test, Y_test,
    embedding=bert_word_to_vec, decoder=index_to_vocab,
    seed_word='Machine', num_gen=10, sample=True,
)

=== accuracy: train / test ===
  model               train     test
  manual             0.0098   0.0100
  pytorch            0.9146   0.8200
  tensorflow         0.9512   0.9000

=== generation from 'Machine' ===
  manual           Machine being performance programmed. performance data. their increasingly their autonomous performance
  pytorch          Machine learning are widely used in various applications such as recommendation
  tensorflow       Machine learning plays an increasingly important role in helping as recommendation
